# Project 2 (Intermediate): Library Management System

**What will you learn in this project?**
- OOP (Object Oriented Programming) — Classes and Objects
- Inheritance (one class using another class's features)
- File handling with JSON (making data persistent)
- Custom Exceptions (creating your own error types)
- Real-world system design thinking

**Think of it like this:**
Imagine you are the **librarian of your school/college library**. You keep records of books, students issue/return them, and late returns get a fine. We will model this exact system using Python classes — just like a real library has separate "things" like Book, Member, and Library, our code will have separate classes for each.


## Step 1: Imports

In [1]:
import json
import os
from datetime import datetime, timedelta

DATA_FILE = "library_data.json"


## Step 2: Creating Custom Exceptions

If someone tries to issue a book that is already issued, we'll show our own clear error instead of a generic Python error. This makes the code more professional and easier to understand.

In [2]:
class BookNotAvailableError(Exception):
    """Raised when a book is already issued and someone tries to issue it again."""
    pass

class MemberNotFoundError(Exception):
    """Raised when a member is not registered in the library."""
    pass


## Step 3: `Book` Class

Every book will have its own properties: title, author, and a unique ID. `is_issued` tells us whether the book is currently with someone or available in the library.

In [3]:
class Book:
    def __init__(self, book_id, title, author):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.is_issued = False
        self.issued_to = None
        self.due_date = None

    def to_dict(self):
        return {
            "book_id": self.book_id,
            "title": self.title,
            "author": self.author,
            "is_issued": self.is_issued,
            "issued_to": self.issued_to,
            "due_date": self.due_date
        }

    def __str__(self):
        status = f"Issued to {self.issued_to}" if self.is_issued else "Available"
        return f"[{self.book_id}] '{self.title}' by {self.author} — {status}"


## Step 4: `Member` Class

Every member (student/user) will have their own record — how many books they've issued, and how much fine they owe.

In [4]:
class Member:
    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name
        self.issued_books = []
        self.fine_due = 0

    def to_dict(self):
        return {
            "member_id": self.member_id,
            "name": self.name,
            "issued_books": self.issued_books,
            "fine_due": self.fine_due
        }

    def __str__(self):
        return f"[{self.member_id}] {self.name} | Books issued: {len(self.issued_books)} | Fine: ₹{self.fine_due}"


## Step 5: `Library` Class — This is where real OOP power shows up

The `Library` class manages both `Book` and `Member`. This class is our **main system** — just like a real librarian controls everything in a real library.

**Fine rule:** Books must be returned within 7 days, after that ₹5 fine per extra day.

In [5]:
class Library:
    FINE_PER_DAY = 5
    ISSUE_DAYS = 7

    def __init__(self):
        self.books = {}
        self.members = {}
        self.load_data()

    # ---------- Book Management ----------
    def add_book(self, book_id, title, author):
        self.books[book_id] = Book(book_id, title, author)
        self.save_data()
        print(f"✅ Book added: {title}")

    def register_member(self, member_id, name):
        self.members[member_id] = Member(member_id, name)
        self.save_data()
        print(f"✅ Member registered: {name}")

    # ---------- Issue / Return Logic ----------
    def issue_book(self, book_id, member_id):
        if book_id not in self.books:
            raise KeyError(f"Book ID '{book_id}' does not exist.")
        if member_id not in self.members:
            raise MemberNotFoundError(f"Member ID '{member_id}' is not registered.")

        book = self.books[book_id]
        if book.is_issued:
            raise BookNotAvailableError(f"'{book.title}' is already issued to '{book.issued_to}'.")

        due = datetime.now() + timedelta(days=self.ISSUE_DAYS)
        book.is_issued = True
        book.issued_to = member_id
        book.due_date = due.strftime("%Y-%m-%d")

        self.members[member_id].issued_books.append(book_id)
        self.save_data()
        print(f"📖 '{book.title}' issued to {self.members[member_id].name}. Return by {book.due_date}")

    def return_book(self, book_id):
        if book_id not in self.books:
            raise KeyError(f"Book ID '{book_id}' does not exist.")

        book = self.books[book_id]
        if not book.is_issued:
            print("ℹ️  This book is already in the library.")
            return

        due_date = datetime.strptime(book.due_date, "%Y-%m-%d")
        days_late = (datetime.now() - due_date).days
        fine = max(0, days_late) * self.FINE_PER_DAY

        member = self.members[book.issued_to]
        member.issued_books.remove(book_id)
        member.fine_due += fine

        member_name = member.name
        book.is_issued = False
        book.issued_to = None
        book.due_date = None

        self.save_data()
        if fine > 0:
            print(f"↩️  Returned. ⚠️  Late return! Fine charged: ₹{fine}")
        else:
            print(f"↩️  '{book.title}' returned on time. No fine. 🎉")

    # ---------- Display ----------
    def show_all_books(self):
        if not self.books:
            print("📭 No books in the library.")
        for book in self.books.values():
            print(book)

    def show_all_members(self):
        if not self.members:
            print("📭 No members registered.")
        for member in self.members.values():
            print(member)

    # ---------- Persistence ----------
    def save_data(self):
        data = {
            "books": {bid: b.to_dict() for bid, b in self.books.items()},
            "members": {mid: m.to_dict() for mid, m in self.members.items()}
        }
        with open(DATA_FILE, "w") as f:
            json.dump(data, f, indent=4)

    def load_data(self):
        if not os.path.exists(DATA_FILE):
            return
        with open(DATA_FILE, "r") as f:
            data = json.load(f)

        for bid, b in data.get("books", {}).items():
            book = Book(b["book_id"], b["title"], b["author"])
            book.is_issued = b["is_issued"]
            book.issued_to = b["issued_to"]
            book.due_date = b["due_date"]
            self.books[bid] = book

        for mid, m in data.get("members", {}).items():
            member = Member(m["member_id"], m["name"])
            member.issued_books = m["issued_books"]
            member.fine_due = m["fine_due"]
            self.members[mid] = member


## Step 6: Demo — Run the system and see it in action

Let's set up a small real-life scene: 2 books, 1 member, issue and return.

In [6]:
library = Library()

library.add_book("B1", "Python Crash Course", "Eric Matthes")
library.add_book("B2", "Atomic Habits", "James Clear")
library.register_member("M1", "Shaurab")

library.show_all_books()
print()
library.issue_book("B1", "M1")
print()
library.show_all_books()


✅ Book added: Python Crash Course
✅ Book added: Atomic Habits
✅ Member registered: Shaurab
[B1] 'Python Crash Course' by Eric Matthes — Available
[B2] 'Atomic Habits' by James Clear — Available

📖 'Python Crash Course' issued to Shaurab. Return by 2026-07-17

[B1] 'Python Crash Course' by Eric Matthes — Issued to M1
[B2] 'Atomic Habits' by James Clear — Available


In [7]:
# Now let's return it (no fine since it was just issued)
library.return_book("B1")
print()
library.show_all_members()


↩️  'Python Crash Course' returned on time. No fine. 🎉

[M1] Shaurab | Books issued: 0 | Fine: ₹0


## Step 7: See exception handling in real use

If someone tries to issue an already-issued book again, our custom error gives a clean message instead of crashing the program.

In [8]:
library.issue_book("B2", "M1")

try:
    library.issue_book("B2", "M1")   # Trying to issue the same book again
except BookNotAvailableError as e:
    print(f"❌ Error handled: {e}")


📖 'Atomic Habits' issued to Shaurab. Return by 2026-07-17
❌ Error handled: 'Atomic Habits' is already issued to 'M1'.
